# Advanced Session: Electrolyte Conductivity Challenge
AI for Materials Science — Hands-on session 5 (optional)

A one-week, Kaggle-style code challenge: predict the **ionic conductivity k (mS/cm)** of liquid electrolytes
from their salt, concentration, solvent mixture and temperature. You get the training table with labels and
upload one Python file, `solution.py`, with a function `fit_predict(train, test)`. The course server runs it on its
GPU (an NVIDIA RTX 5060): it trains on the training table, predicts the hidden test rows and scores the predictions
against hidden labels.

**Site:** https://hos5.tail1aab16.ts.net

- Task: predict ionic conductivity k (mS/cm) of liquid electrolytes, CALiSol-23 test split. Metric: RMSLE, the RMSE between ln(1 + predicted k) and ln(1 + true k) on the hidden labels (lower is better); negative predictions count as 0.
- Public leaderboard: 30% of the test formulations; private: the remaining 70%, revealed after the deadline.
- Open: 2026-11-02 17:30 KST. Deadline: 2026-11-08 23:59 KST. Private results: 2026-11-09 09:00 KST.
- Submission: one Python file, solution.py, defining fit_predict(train, test) that returns one predicted k per test row, in test-row order. train has the target column k; test has the same feature columns without k.
- The course server runs it on one NVIDIA RTX 5060 GPU (8 GB), without internet access, within 3 minutes, with 2 CPU cores and 8 GB of memory. Libraries: numpy, pandas, scipy, scikit-learn, torch, xgboost, lightgbm.
- Compute on the GPU: PyTorch models and tensors .to("cuda"), XGBoost device="cuda". scikit-learn models run on the 2 CPU cores only.
- Limit: 5 scored submissions per student per KST day; runs that fail do not count, up to 10 failed runs a day. The upload time decides whether a submission made the deadline.
- Identity: your student ID and the 6-digit access code sent to you on KLMS. Only your nickname is shown.
- Final ranking uses the private RMSLE of each student's best public submission.
- Do not look up or reuse the original CALiSol-23 values for the test formulations; fit_predict must learn from the train table it receives. Suspicious entries are flagged and may be disqualified.
- The TA reads the code of the top 3 submissions before results are announced in class on Nov 9.
- This is an optional activity; any bonus credit is decided by the instructor.
- Too many failed logins trigger a 15-minute cool-down; 30 failures in a day lock that ID for the day.
- Locked? Contact the TA; a server restart clears locks.

---

Data: CALiSol-23, de Blasio et al., Sci. Data 11, 750 (2024), CC BY 4.0 (modified: cleaned and split).

**First:** File ▸ Save a copy in Drive, and work in your copy.

Run the cells one at a time from the top. Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not defined" error. Lines starting with `##` inside the code are comments written for you; Python does not run them.

**Shared or lab PC?** Sign out of Google when you finish and choose Runtime ▸ Disconnect and delete runtime.

## 0. Setup

### 0-1. Check the libraries
Nothing to install: pandas, scikit-learn, XGBoost, PyTorch and Matplotlib ship with Colab. The server has the
libraries listed in the rules above; a model that runs here with them runs there too. For a GPU in Colab choose
Runtime ▸ Change runtime type ▸ GPU; the code below also runs without one.

In [ ]:
import pandas, sklearn, torch, xgboost
print("pandas", pandas.__version__, " scikit-learn", sklearn.__version__, " xgboost", xgboost.__version__,
      " torch", torch.__version__, " GPU", torch.cuda.is_available())

### 0-2. Download the course material

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries

In [ ]:
import importlib.util, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   ## the GPU when there is one (the server has one)
np.random.seed(0)
torch.manual_seed(0)

### 0-4. Set the file paths

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
print("Data folder exists:", DATA_DIR.exists())

## A. The data

### A-1. Load the training table
`train` has the target column `k` and the feature columns. The test rows stay on the server: your
`fit_predict` receives them as a table with the same feature columns and no `k`.

In [ ]:
train = pd.read_csv(DATA_DIR / "calisol23_train.csv")
print("train:", train.shape)
train.head()

### A-2. What the columns mean
- `T` — temperature in kelvin.
- `c` — lithium-salt concentration, in the unit given by `c units` (mol/kg or mol/L).
- `salt` — the lithium salt (LiPF6, LiBF4, LiBOB, ...).
- `solvent ratio type` — whether the solvent columns are weight (`w`), volume (`v`) or mole (`mol`) fractions.
- 38 solvent columns (EC, PC, DMC, EMC, DEC, DME, ...) — the fraction of each solvent; each row sums to 1.
- `k` — ionic conductivity in mS/cm (train only). Most values are below 15 mS/cm.

The hidden test rows are split into a public part (shown on the leaderboard during the week) and a private
part (final ranking). Every formulation appears on one side only, so the test rows are new
salt/solvent/concentration combinations, not new temperatures of a known mixture.

### A-3. The target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(train["k"], bins=60)
ax.set_xlabel("ionic conductivity k (mS/cm)"); ax.set_ylabel("rows")
ax.set_title("Training set")
fig.savefig(OUTPUT / "A3_k_hist.png", dpi=150, bbox_inches="tight")
plt.show()
print(train["k"].describe().round(3))

## B. Features
Three columns are text (categories) and the rest are numbers. Most models need numbers only, so how you turn the
text columns into numbers is part of your model: fit that step on the training rows only, and make sure a
category that appears only in the test rows does not crash it.

In [ ]:
CATEGORICAL = ["salt", "c units", "solvent ratio type"]
NUMERIC = [c for c in train.columns if c not in CATEGORICAL + ["k"]]
X, y = train.drop(columns="k"), train["k"]
print(len(NUMERIC), "numeric columns +", len(CATEGORICAL), "categorical columns")

## C. Validate the way the leaderboard scores

### C-1. Formulations as groups
The leaderboard's hidden split keeps every row of one formulation (salt, concentration, unit, ratio type and
the 38 solvent fractions) on one side. Validate the same way, or your local score will be too optimistic.

In [ ]:
KEY = ["salt", "c", "c units", "solvent ratio type"] + [c for c in NUMERIC if c not in ("T", "c")]
groups = train[KEY].astype(str).agg("|".join, axis=1)
print(groups.nunique(), "formulations in", len(train), "training rows")

### C-2. A validation tool
Which model to use is your decision: a decision tree, a random forest, an MLP (scikit-learn or PyTorch), XGBoost,
LightGBM, or anything else built from the libraries in the rules. `grouped_rmsle(make_model, X, y, groups)` scores
a model the way the leaderboard does: 5 folds, every formulation on one side, RMSLE (the RMSE of ln(1 + k),
negative predictions counted as 0) averaged over the folds. On this scale the few large k values no longer
dominate the score, so low-conductivity electrolytes count too. `make_model` is a function that returns a new,
untrained model with `fit(X, y)` and `predict(X)`, for example `lambda: make_pipeline(...)`: every fold builds its own
model, so nothing learned on one fold leaks into the score of another. Put the text-column encoding inside the
model (for example a scikit-learn pipeline) so that every fold fits it on its own training rows.

In [ ]:
def grouped_rmsle(make_model, X, y, groups, n_splits=5):
    scores = []
    for fit_idx, val_idx in GroupKFold(n_splits=n_splits).split(X, y, groups=groups):
        model = make_model()   ## a new, untrained model for every fold
        model.fit(X.iloc[fit_idx], y.iloc[fit_idx])
        pred = np.clip(model.predict(X.iloc[val_idx]), 0, None)
        scores.append(np.sqrt(mean_squared_error(np.log1p(y.iloc[val_idx]), np.log1p(pred))))
    return float(np.mean(scores)), float(np.std(scores))

### C-3. Practice 1
Choose two or more candidate models, build each one with its own text-column encoding, and compare them with
`grouped_rmsle`. Then tune the best one with the same function, never with the leaderboard. The server allows
3 minutes for one training run plus the prediction, so time one fit of your final model.

In [ ]:
## Write your code here.
## Order: write make_model() -> mean, std = grouped_rmsle(make_model, X, y, groups) -> compare -> tune the best one

## D. Make a submission

### D-1. Write solution.py
The server imports this one file and calls `fit_predict(train, test)`: `train` is the whole training table
(with `k`), `test` the hidden test rows (the same feature columns, no `k`). Return one predicted `k` per test
row, in test-row order. The file runs on its own on the server, not inside this notebook, so it must import
everything it uses and do its own text-column encoding. The cell below writes an empty template: write your
model from C-3 into `fit_predict` and run the cell again. Running the cell writes the file; it does not run it.

In [ ]:
%%writefile solution.py
## solution.py: your submission. Import here everything fit_predict uses.


def fit_predict(train, test):
    ## train: the training table with the target column k. test: the same feature columns without k.
    ## Return one predicted k per test row, in test-row order.
    raise NotImplementedError("write your model in fit_predict (D-1) and run the D-1 cell again")

### D-2. Test it the way the server runs it
This cell imports `solution.py` from the file, holds out 20% of the formulations, and calls `fit_predict` on
the rest exactly as the server will. It checks what the server checks (one finite number per test row) and
times the call: the server allows 3 minutes. If this cell fails, the server run fails too. While
`solution.py` is still the empty template, the cell says so instead.

In [ ]:
def load_solution(path="solution.py"):
    spec = importlib.util.spec_from_file_location("solution", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)   ## a fresh import: the server sees only this file
    return module

solution = load_solution()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
fit_idx, hold_idx = next(splitter.split(train, groups=groups))
part, hold = train.iloc[fit_idx], train.iloc[hold_idx]
t0 = time.time()
try:
    pred = np.asarray(solution.fit_predict(part, hold.drop(columns="k")), dtype=float).reshape(-1)
except NotImplementedError as err:
    print("solution.py is still the empty template:", err)
else:
    seconds = time.time() - t0
    assert len(pred) == len(hold) and np.isfinite(pred).all(), "the server would reject these predictions"
    rmsle = np.sqrt(mean_squared_error(np.log1p(hold["k"]), np.log1p(np.clip(pred, 0, None))))
    print(f"hold-out RMSLE {rmsle:.3f} in {seconds:.0f} s")

### D-3. Submit
1. Run the cell below: in Colab it downloads `solution.py` to your computer (the site's file picker reads your
   computer, not the Colab runtime). You can also download it from Colab's Files panel.
2. Open the site linked at the top of this notebook, enter your student ID and the 6-digit code from KLMS, and
   upload `solution.py`. The receipt shows your place in the queue.
3. Press **My submissions** a few minutes later: it shows the status, your public RMSLE or the error message
   of a failed run, and how many of your 5 submissions you have left today. Failed runs do not
   count against that limit.

In [ ]:
try:   ## in Colab: download the file to your computer, then upload it on the site
    from google.colab import files
    files.download("solution.py")
except ImportError:
    print("Not in Colab: solution.py is in the working folder.")

## E. Ideas to try
- Train on `np.log1p(k)`, the scale the metric uses, and turn the predictions back with `np.expm1`;
  conductivity spans two orders of magnitude.
- Add `1/T` as a feature: conductivity follows an Arrhenius-like temperature dependence.
- Merge salt synonyms (`LiTFSI` and `LiN(CF3SO2)2` are the same salt) and group rare salts.
- Use the GPU for the heavy part: PyTorch models and tensors move there with `.to("cuda")`, XGBoost takes
  `device="cuda"`; on the server's RTX 5060 a wider network or more trees cost seconds.
- Try a different model family: the best one for this table is not given in advance.

### E-2. Practice 2
Implement one idea above, measure it with `grouped_rmsle`, and if it beats your current model put it into
`fit_predict` (edit the D-1 cell and run it again), run D-2, and submit the new `solution.py`.

In [ ]:
## Write your code here.
## Order: change make_model or the features -> grouped_rmsle(make_model, ...) -> compare with your current model -> update solution.py (D-1) -> D-2 -> submit

## Data sources

### CALiSol-23
`Data/calisol23_train.csv` (the test rows are kept on the course server). P. de Blasio, J. Elsborg, T. Vegge, E. Flores, A. Bhowmik, CALiSol-23: Experimental electrolyte conductivity data for various Li-salts and solvent combinations, *Scientific Data* 11, 750 (2024), https://doi.org/10.1038/s41597-024-03575-8. Dataset: DTU Data, https://doi.org/10.11583/DTU.c.6929599.v1, licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Changes: 556 rows removed (missing or non-positive k, negative c); whitespace stripped; doi column removed; split into train/test by formulation; the test rows (features and labels) stay on the course server for the competition.